In [2]:
import math

class ConvexPolygonLocator:
    def __init__(self, polygon):
        cx = sum(p[0] for p in polygon) / len(polygon)
        cy = sum(p[1] for p in polygon) / len(polygon)
        self.center = (cx,cy)

        self.poly = sorted(polygon,
            key = lambda p: math.atan2(p[1] - cy, p[0] - cx))
        self.angles = [math.atan2(p[1] - cy, p[0] - cx)
                      for p in self.poly]


    def _cross(self, o, a, b):
        return(a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])

    def contains(self, point):
        cx, cy = self.center
        angle = math.atan2(point[1] - cy, point[0] - cx)

        import bisect
        i = bisect.bisect_right(self.angles, angle) - 1
        i %= len(self.poly)
        j = (i + 1) % len(self.poly)

        d1 = self._cross(self.center, self.poly[i], point)
        d2 = self._cross(self.poly[i], self.poly[j], point)
        d3 = self._cross(self.poly[j], self.center, point)
        return (d1 >= 0 and d2 >= 0 and d3 >= 0) or \
               (d1 <= 0 and d2 <= 0 and d3 <= 0)


hexagon = [(2,0), (1,1.7), (-1,1.7),
           (-2,0),(-1,-1.7), (1,-1.7)]


locator = ConvexPolygonLocator(hexagon)
print(locator.contains((0,0)))
print(locator.contains((3,3)))

True
False


In [3]:
import random

class Trapezoid:
    def __init__(self, top, bottom, left, right, face):
        self.top, self.bottom = top, bottom
        self.left, self.right = left, right
        self.face = face
        self.left_p = self.right_p = None


class DAGNode:
    def __init__(self, kind, data, left = None, right =None):
        self.kind = kind
        self.data = data
        self.left , self.right = left, right


def locate(node, point):
    while node.kind != 'leaf':
        if node.kind == 'x':
            node = node.left if point[0] < node.data[0] else node.right
        else:
            (x1,y1), (x2,y2) = node.data
            side = (x2 - x1)*(point[1]-y1) - (y2 - y1)*(point[0] - x1)
            node = node.left if side > 0 else node.right
    return node.data.face




In [5]:
from shapely.geometry import Point, Polygon
from shapely.strtree import STRtree

region = [
    Polygon([(0,0), (5,0), (5,5), (0,5)]),
    Polygon([(5,0),(10,0),(10,5),(5,5)]),
    Polygon([(0,5),(10,5),(10,10),(0,10)]),
]

tree = STRtree(region)
def locate_region(point, tree, regions_param):
    p = Point(point)
    candidates = tree.query(p)
    for idx in candidates:
        if regions_param[idx].contains(p):
            return idx
    return None


print(locate_region((3,3), tree, region))
print(locate_region((7,8), tree, region))

from scipy.spatial import KDTree
import numpy as np

sites = np.array([[2,2], [8,2], [5,8]])
tree_v = KDTree(sites)

def locate_voronoi_cell(point):
    dist, idx = tree_v.query(point)
    return idx



print(locate_voronoi_cell([4,3]))

0
2
0


In [ ]:
def cross2d(o,a,b):
    return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])


def on_segment(p,a,b):
    return (min(a[0],b[0]) <= p[0] <= max(a[0],b[0]) and
            min(a[1],[b[1]]) <= p[1] <= max(a[1],b[1]))
